#  Comparison: Own measurements vs MagNet vs datasheet
- 100 kHz to 500 kHz (max frequency, where measurements are present from MagNet database at all temperatures)
- Steinmetz fit with quadratic temperature term

In [ ]:
# general imports
import logging
from itertools import product
from typing import cast, Union, List, Dict, Tuple
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
import pandas as pd
import itertools
import numpy as np

# imports from this repo
from meta import paths

# imports from external but local repo
import materialdatabase as mdb
from materialdatabase import get_user_colors as colors
from materialdatabase.processing.plot import StyleDict, _flatten_y_columns
from materialdatabase.meta.data_classes import ComplexPermeabilityPlotConfig, ComplexPermeabilityConfig
from materialdatabase.processing.utils.math import mean_relative_absolute_error as mre

logging.basicConfig(format='%(levelname)s: %(message)s', level=logging.INFO)

# -------------------------
# Matplotlib settings
# -------------------------
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.sans-serif": ["Bitstream Vera Sans", "DejaVu Sans", "Helvetica", "Arial", "sans-serif"],
    "font.size": 9.0,
    "text.latex.preamble": r"\usepackage{upgreek}\usepackage{siunitx}",
    "mathtext.fontset": "custom",
    "mathtext.rm": "Bitstream Vera Serif",
    "mathtext.it": "Bitstream Vera Serif:italic",
    "mathtext.bf": "Bitstream Vera Serif:bold"
})


# ---------------------------------------------
# Configuration
# ---------------------------------------------
cm = 1 / 2.54

# data limits for Steinmetz fitting:
# the MagNet data only has data points for measurements up to 500 kHz at all temperatures
f_min = 90e3
f_max = 1300e3

# select fit function (for a rather small frequency range, classical Steinmetz is sufficient)
pv_fit_function = mdb.FitFunction.enhancedSteinmetz

# Operating points of interest
FREQS = np.linspace(100e3, 1000e3, 10)  # Frequency in Hz
TEMPS = [60]  # Temperature in °C
FLUX_DENSITIES = [50e-3, 75e-3, 100e-3]  # Flux density in T


# Materials to compare
max_frequencies_config = {
    "N49_LEA_probe_1": 1.3e6,
    "N49_LEA_probe_2": 1.3e6,
    "N49_TDK": 1e6,
    "N49_MagNet": 500e3,
}
materials_config: dict[str, ComplexPermeabilityPlotConfig] = {
    "N49_LEA_probe_1": ComplexPermeabilityPlotConfig(
        mat_cfg=ComplexPermeabilityConfig(
            material=mdb.Material.N49,
            setup=mdb.DataSource.LEA_MTB,
            pv_fit_function=pv_fit_function,
            probe_codes=["65Y"]
        ),
        enabled=True,
        label="meas. (R22.1x13.7x7.9)",
        color=colors().compare1,
        marker="*"
    ),
    "N49_LEA_probe_2": ComplexPermeabilityPlotConfig(
        mat_cfg=ComplexPermeabilityConfig(
            material=mdb.Material.N49,
            setup=mdb.DataSource.LEA_MTB,
            pv_fit_function=pv_fit_function,
            probe_codes=["R16x9.6x6.3"]
        ),
        enabled=True,
        label="meas. (R16x9.6x6.3)",
        color=colors().compare1,
        marker="^"
    ),
    "N49_TDK": ComplexPermeabilityPlotConfig(
        mat_cfg=ComplexPermeabilityConfig(
            material=mdb.Material.N49,
            setup=mdb.DataSource.TDK_MDT,
            pv_fit_function=pv_fit_function,

        ),
        enabled=True,
        label="datasheet",
        color=colors().gtruth,
        marker="."
    ),
    "N49_MagNet": ComplexPermeabilityPlotConfig(
        mat_cfg=ComplexPermeabilityConfig(
            material=mdb.Material.N49,
            setup=mdb.DataSource.MagNet,
            pv_fit_function=pv_fit_function
        ),
        enabled=True,
        label="MagNet (R16x9.6x6.3)",
        color=colors().compare2,
        marker="."
    )
}

# ---------------------------------------------
# Load Material Data and Prepare Grid
# ---------------------------------------------
mdb_data = mdb.Data()

# Create sweep grid of all combinations
df_common = pd.DataFrame(
    product(FREQS, TEMPS, FLUX_DENSITIES),
    columns=["f", "T", "b"]
)

# ---------------------------------------------
# Plot function
# ---------------------------------------------
def plot_loss_vs_frequency(ax: Axes, df: pd.DataFrame,
                           y_columns: Union[List[str], tuple], styles: Dict[str, StyleDict]) -> None:
    """
    Plot core loss density versus frequency for multiple data series.

    :param ax: Matplotlib Axes object to plot on
    :param df: DataFrame with columns ['f', 'b', 'T'] plus loss columns
    :param y_columns: List or tuple of column names for y-axis data (loss columns)
    :param styles: Dictionary mapping y_column to dict with keys 'marker', 'color', 'label'
    :param annotate: If True, annotate final data points with (b, T) values
    :param y_label: If True, plot the y_label
    :param connect_all: If True, draw a line connecting all points of each y_column
    """
    y_columns = _flatten_y_columns(y_columns)

    # Precompute frequency in kHz for the raw scatter plot
    f_kHz = df["f"] / 1000

    for i_col, y_col in enumerate(y_columns):
        style = styles[y_col]

        # Plot individual points
        ax.loglog(f_kHz, df[y_col] / 1000,
                  style["marker"],
                  color=style["color"],
                  label=style["label"])

        # Plot lines for each (b, T) group
        for key, group in df.groupby(["b", "T"]):
            b, T = cast(Tuple[float, float], key)
            sorted_group = group.sort_values("f")
            f_vals = sorted_group["f"] / 1000
            y_vals = sorted_group[y_col] / 1000

            ax.loglog(f_vals, y_vals, "-", color=style["color"], alpha=0.7)

            if i_col == 0:
                x_pos = f_vals.iloc[0]
                y_pos = y_vals.iloc[0]

                ax.text(
                    x_pos,
                    y_pos * 1.1,
                    f"{int(b * 1000)} mT",
                    fontsize=8,
                    color="k",
                    ha="left",
                    va="bottom",
                    alpha=1,
                    bbox=dict(facecolor=(1, 1, 1, 0.5), edgecolor="k", linewidth=1)
                )

    ax.set_xlabel(r"$f$ / $\mathrm{kHz}$")
    ax.set_ylabel(r"$p_\mathrm{v}$ / $\mathrm{kW}/\mathrm{m}^3$")

    ax.grid(True, which="both")
    ax.legend(loc="lower center", ncol=2, bbox_to_anchor=(0.5, 1), fontsize=7, frameon=True) #, title=r"N49 loss data at \qty{60}{\celsius} from:")

    ax.text(
        100,
        3000,
        r"N49 loss data at \qty{60}{\celsius}",
        fontsize=8,
        color="k",
        ha="left",
        va="top",
        alpha=1,
        bbox=dict(facecolor=(1, 1, 1, 1), edgecolor="k", linewidth=1)
    )

    # -------------------------
    # Save + show
    # -------------------------
    plt.tight_layout()
    plt.savefig(paths.grafics.joinpath("measurement_vs_MagNet.pdf"))
    plt.show()


# ---------------------------------------------
# Plot Core Loss
# ---------------------------------------------

param_storage = {}
styles_pv = {}
for key, cfg in materials_config.items():
    if not cfg.enabled:
        continue

    logging.info(f"---")
    logging.info(f"Fitting power loss for: {cfg.label} ({cfg.mat_cfg.setup.name})")

    material = mdb_data.get_complex_permeability(
        material=cfg.mat_cfg.material,
        data_source=cfg.mat_cfg.setup,
        pv_fit_function=cfg.mat_cfg.pv_fit_function,
        probe_codes=cfg.mat_cfg.probe_codes
    )

    # visualize data distribution -> investigate data imbalance
    # fig, axs = plt.subplots(1, 1, )
    # # plt.scatter(material.measurement_data["f"], material.measurement_data["b"])
    # plt.hist(material.measurement_data["f"], bins=20)
    # plt.title(f"{cfg.label} ({cfg.mat_cfg.setup.name})")
    # plt.show()

    # Fit losses and store fit parameters
    param_storage[f"{cfg.label}"] = material.fit_losses(f_min=f_min,
                                                        f_max=f_max,
                                                        b_max=120e-3)  # avoid data imbalance (all datasets have more data points and larger flux densities at low frequencies)

    # Apply per-material frequency limit before computing losses
    f_limit = max_frequencies_config.get(key, f_max)
    df_filtered = df_common[df_common["f"] <= f_limit].copy()

    col_name = f"pv_{key}"
    df_filtered[col_name] = material.pv_fit_function.get_function()(
        (df_filtered["f"].to_numpy(),
         df_filtered["T"].to_numpy(),
         df_filtered["b"].to_numpy()),
        *material.params_pv
    )

    # merge results into original grid (optional NaN fill for missing high-f reqs)
    df_common = df_common.merge(
        df_filtered[["f", "T", "b", col_name]],
        on=["f", "T", "b"],
        how="left"
    )

    styles_pv[col_name] = cast(StyleDict, {
        "marker": cfg.marker,
        "color": cfg.color,
        "label": cfg.label
    })

print(param_storage)

fig, axs = plt.subplots(1, 1, figsize=(9 * cm, 8 * cm))
plot_loss_vs_frequency(ax=axs, df=df_common, y_columns=list(styles_pv.keys()), styles=styles_pv)


In [ ]:
# Define the probe columns
cols = ["pv_N49_TDK", "pv_N49_LEA_probe_1", "pv_N49_LEA_probe_2", "pv_N49_MagNet"]

# Initialize empty DataFrame for cross‑error matrix (in %)
error_table = pd.DataFrame(index=cols, columns=cols, dtype=float)

# Fill the matrix
for c1, c2 in itertools.product(cols, repeat=2):
    error_table.loc[c1, c2] = mre(df_common[c1], df_common[c2]) * 100  # convert to %

# Set diagonal to 0%
np.fill_diagonal(error_table.values, 0)

# Optionally round and pretty‑print
error_table = error_table.round(1)
print(error_table)